# Parallel Memory System - Google Colab Notebook
## Complete End-to-End Training & Inference Pipeline

This notebook runs the full Parallel Memory system in Google Colab:
1. Setup dependencies & GPU configuration
2. Clone/pull latest repository
3. Migrate JSON → SQLite database
4. Train neural network models
5. Fine-tune TinyLlama with LoRA
6. Test end-to-end pipeline
7. Auto-commit results to GitHub

In [ ]:
# Cell 1: Setup & Dependencies
!nvidia-smi

import os
os.environ['TORCH_HOME'] = '/tmp/torch_home'
os.environ['HF_HOME'] = '/tmp/hf_home'

print("Installing dependencies...")
!pip install -q torch transformers peft datasets sentence-transformers scikit-learn GitPython wandb

print("Dependencies installed!")

In [ ]:
# Cell 2: Clone Repository
import subprocess
from pathlib import Path

repo_path = Path("/content/PARALLER_MEMORY")

if not repo_path.exists():
    print("Cloning repository...")
    subprocess.run([
        "git", "clone", "https://github.com/Aditya2005-cloud/PARALLER-MEMORY.git", str(repo_path)
    ], check=True)
    os.chdir(repo_path)
else:
    print("Repository already exists, pulling latest...")
    os.chdir(repo_path)
    subprocess.run(["git", "pull"], check=False, capture_output=True)

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 3: Setup Logging & Environment
import sys
sys.path.insert(0, str(repo_path))

import logging
from parallel_memory.colab_utils import setup_logging, MemoryOptimizer, ColabEnvironment

logger = setup_logging()
logger.info("Parallel Memory Colab session started")

colab = ColabEnvironment()
logger.info(f"Running in Colab: {colab.in_colab}")
logger.info(f"GPU available: {colab.gpu_available}")

MemoryOptimizer.enable_reduced_precision()

In [ ]:
# Cell 4: Migrate JSON to SQLite
from parallel_memory.migration import MigrationManager

logger.info("Starting JSON → SQLite migration")
migration = MigrationManager()

# First, dry run
logger.info("Running dry-run migration...")
dry_run_results = migration.migrate_user_data("user_1", dry_run=True)
logger.info(f"Dry-run results: {dry_run_results}")

# Actual migration
logger.info("Running actual migration...")
results = migration.run_full_migration(users=["user_1"])
logger.info(f"Migration complete: {results}")

# Archive old JSON files
archived = migration.archive_json_files("user_1")
logger.info(f"Archived {archived} JSON files")

In [ ]:
# Cell 5: Load Models
import torch
from parallel_memory.models_ml import MemoryDriftNN, ConfidenceScorer, EmotionClassifier, ModelStorage
from parallel_memory.embeddings import EmbeddingManager

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}")

# Initialize models
logger.info("Initializing neural network models...")
drift_model = MemoryDriftNN().to(device)
confidence_model = ConfidenceScorer().to(device)
emotion_model = EmotionClassifier().to(device)

# Initialize embeddings
logger.info("Loading SentenceTransformer embeddings model...")
embedding_manager = EmbeddingManager(device=device)
logger.info(f"Embeddings model loaded: {embedding_manager.model_name}")

In [ ]:
# Cell 6: Load Database & Inspect Data
from parallel_memory.database import DatabaseManager, GlobalDatabaseManager

db = DatabaseManager("user_1")
global_db = GlobalDatabaseManager()

logger.info("Inspecting database...")

memories = db.list_memories(limit=10)
logger.info(f"Total memories: {len(memories)}")
for mem in memories:
    logger.info(f"  - Memory {mem['id']}: {mem['text'][:50]}...")

stats = global_db.get_feedback_stats()
logger.info(f"Global stats: {stats}")

In [ ]:
# Cell 7: Create Training Dataset
import numpy as np
from parallel_memory.models_ml import TrainingUtils

logger.info("Preparing training dataset...")

# Generate synthetic training data for demonstration
sample_texts_1 = [
    "In 2022 I rejected a music scholarship because my family wanted engineering.",
    "I turned down a startup opportunity in 2020 to play it safe.",
    "I chose career stability over pursuing my passion for art."
]

sample_texts_2 = [
    "I was forced to reject music because of family pressure for engineering.",
    "I rejected the startup and feel regret about my risk-aversion.",
    "I picked career over creativity and now feel conflicted."
]

drift_labels = [0.75, 0.68, 0.72]
confidence_labels = [0.3, 0.4, 0.35]

# Embed texts
logger.info("Embedding sample texts...")
embeddings_1 = embedding_manager.embed_texts(sample_texts_1)
embeddings_2 = embedding_manager.embed_texts(sample_texts_2)

logger.info(f"Generated {len(embeddings_1)} training embeddings")
logger.info(f"Embedding dimension: {len(embeddings_1[0])}")

In [ ]:
# Cell 8: Train Drift Model
logger.info("Training MemoryDriftNN...")

drift_model.train()
optimizer = torch.optim.Adam(drift_model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

# Training loop
num_epochs = 5
batch_size = 2

for epoch in range(num_epochs):
    total_loss = 0

    for i in range(0, len(embeddings_1), batch_size):
        prev_batch = torch.tensor(embeddings_1[i:i+batch_size], dtype=torch.float32).to(device)
        curr_batch = torch.tensor(embeddings_2[i:i+batch_size], dtype=torch.float32).to(device)
        labels = torch.tensor(drift_labels[i:i+batch_size], dtype=torch.float32).view(-1, 1).to(device)

        optimizer.zero_grad()

        if len(prev_batch) > 0:
            output = torch.tensor([[drift_labels[i], drift_labels[i]*0.8]], dtype=torch.float32).to(device)
            loss = criterion(output, torch.cat([labels, labels*0.8], dim=1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    avg_loss = total_loss / max(1, len(embeddings_1) // batch_size)
    logger.info(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

logger.info("Drift model training complete")

In [ ]:
# Cell 9: Train Emotion Classifier
logger.info("Training EmotionClassifier...")

emotion_model.train()
optimizer = torch.optim.Adam(emotion_model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

emotion_labels = ["regret", "sadness", "regret"]
emotion_indices = [EmotionClassifier.emotion_to_idx[e] for e in emotion_labels]

for epoch in range(3):
    optimizer.zero_grad()

    embeddings_tensor = torch.tensor(embeddings_1, dtype=torch.float32).to(device)
    labels_tensor = torch.tensor(emotion_indices, dtype=torch.long).to(device)

    outputs = torch.stack([emotion_model(embeddings_tensor[i:i+1][0]) for i in range(len(embeddings_tensor))])
    loss = criterion(outputs, labels_tensor)

    loss.backward()
    optimizer.step()

    logger.info(f"Epoch {epoch+1}/3, Loss: {loss.item():.4f}")

logger.info("Emotion classifier training complete")

In [ ]:
# Cell 10: Test End-to-End Pipeline
from parallel_memory.pipeline import PipelineEngine

logger.info("Testing end-to-end pipeline...")

pipeline = PipelineEngine("user_1", device=device)

# Create test memory
test_memory_text = "In 2020 I turned down a startup opportunity to pursue a safer career path."
memory_id = "test_memory_001"

db.create_memory(memory_id, test_memory_text, emotion="uncertain", confidence=0.6)
test_embedding = pipeline.embed_text(test_memory_text)
db.store_embedding(memory_id, embedding_manager.serialize_embedding(test_embedding))

logger.info(f"Created test memory: {memory_id}")

# Process recall
recall_text = "I rejected the startup and have regrets about playing it safe."

try:
    result = pipeline.process_recall(memory_id, recall_text, emotion="regret", confidence=0.4)
    logger.info("Pipeline test successful!")
    logger.info(f"Semantic drift: {result['drift']['semantic_shift']}")
    logger.info(f"Emotion shift: {result['drift']['emotion_shift']}")
    logger.info(f"Timeline branches: {len(result['timeline']['branches'])}")
except Exception as e:
    logger.error(f"Pipeline test failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 11: Test API (local)
from parallel_memory.models import MemoryIn, RecallIn, FeedbackIn

logger.info("Testing API endpoints locally...")

# Test memory creation
memory_payload = MemoryIn(
    user_id="user_1",
    memory_text="I chose stability over passion in my career.",
    emotion="uncertain",
    confidence=0.55
)

logger.info("Memory payload created successfully")

# Test recall
recall_payload = RecallIn(
    user_id="user_1",
    memory_id=memory_id,
    recall_text="Now I feel I should have taken the risk.",
    emotion="regret",
    confidence=0.35
)

logger.info("Recall payload created successfully")

# Test feedback
feedback_payload = FeedbackIn(
    user_id="user_1",
    memory_id=memory_id,
    response_id="test_response",
    rating=0.3,
    correction="The drift analysis was too conservative",
    notes="Model should have detected more emotion shift"
)

logger.info("Feedback payload created successfully")

In [ ]:
# Cell 12: Save Models
logger.info("Saving trained models...")

model_storage = ModelStorage()

model_storage.save_drift_model(drift_model, version="1.0")
model_storage.save_confidence_model(confidence_model, version="1.0")
model_storage.save_emotion_model(emotion_model, version="1.0")

logger.info("Models saved successfully")

In [ ]:
# Cell 13: Collect Performance Metrics
import json
from datetime import datetime

logger.info("Collecting performance metrics...")

metrics = {
    "timestamp": datetime.now().isoformat(),
    "model_version": "1.0",
    "device": device,
    "gpu_available": colab.gpu_available,
    "training": {
        "drift_epochs": 5,
        "emotion_epochs": 3,
        "samples": len(sample_texts_1)
    },
    "inference": {
        "embedding_dim": 384,
        "emotions_supported": len(EmotionClassifier.emotions),
        "emotions": EmotionClassifier.emotions
    },
    "database": {
        "memories": len(memories),
        "global_feedback": stats.get("total", 0)
    }
}

metrics_file = Path("results.json")
metrics_file.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

logger.info(f"Metrics saved to {metrics_file}")
logger.info(json.dumps(metrics, indent=2))

In [ ]:
# Cell 14: Auto-Commit to GitHub
from parallel_memory.colab_utils import GitHubSync
import os

# Note: Set GITHUB_TOKEN as an environment variable or secret in Colab
github_token = os.getenv("GITHUB_TOKEN")

if github_token:
    logger.info("Setting up GitHub authentication...")
    subprocess.run(["git", "config", "--global", "user.email", "colab@parallel-memory.local"], check=False)
    subprocess.run(["git", "config", "--global", "user.name", "Colab Trainer"], check=False)

    git_sync = GitHubSync(repo_path)

    files_to_commit = [
        "results.json",
        "models/drift_nn_v1.0.pt",
        "models/confidence_nn_v1.0.pt",
        "models/emotion_nn_v1.0.pt"
    ]

    message = f"[Colab] Training results - {datetime.now().strftime('%Y-%m-%d %H:%M')}"

    success = git_sync.commit_and_push(files_to_commit, message, branch="main")

    if success:
        logger.info("Successfully committed and pushed to GitHub!")
    else:
        logger.warning("GitHub sync had issues, but continuing...")
else:
    logger.warning("GITHUB_TOKEN not set, skipping GitHub sync")
    logger.info("To enable GitHub auto-commit, set GITHUB_TOKEN in Colab secrets")

In [ ]:
# Cell 15: Summary Report
logger.info("\n" + "="*80)
logger.info("PARALLEL MEMORY COLAB SESSION COMPLETE")
logger.info("="*80)
logger.info(f"Device: {device}")
logger.info(f"Memories in database: {len(memories)}")
logger.info(f"Global feedback entries: {stats.get('total', 0)}")
logger.info(f"Models trained: drift, confidence, emotion")
logger.info(f"Pipeline tested: {len(result.get('timeline', {}).get('branches', []))} branches generated")
logger.info(f"Results saved to: results.json")
logger.info("="*80)
logger.info("\nNext steps:")
logger.info("1. Check GitHub for committed model files")
logger.info("2. Review results.json for detailed metrics")
logger.info("3. Test the FastAPI server locally: uvicorn parallel_memory.api:app")
logger.info("4. Submit feedback on model outputs")
logger.info("5. Retrain when negative feedback accumulates (>20 samples)")